# AutoPose — GigaPose-style 6-DoF Object Pose Estimation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arunachaleshwaran/autopose/blob/main/notebooks/autopose_colab.ipynb)

## Pipeline Overview

| Stage | What | How |
|---|---|---|
| **1 — Template retrieval** | Predict azimuth + elevation (2 DoF) | Nearest-neighbour search in Fae (DINOv2) feature space over 162 icosphere templates |
| **2 — In-plane refinement** | Predict in-plane rotation + translation + scale (4 DoF) | Fist (ResNet18 + MLP heads) regresses (cos α, sin α, tx, ty, s) |

Azimuth & elevation are **retrieved, not regressed** — the best-matching template's pose from `templates/poses.json` is the answer.

---

**Before running:** upload your local `templates/` folder (162 PNGs + `poses.json`) to Google Drive at `MyDrive/autopose/templates/`.

## 0 — Setup

In [ ]:
# Clone repo (skip if already present)
import os

REPO_DIR = "/content/autopose"

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/arunachaleshwaran/autopose.git {REPO_DIR}
else:
    print("Repo already cloned, pulling latest...")
    !git -C {REPO_DIR} pull --ff-only

%cd {REPO_DIR}

In [ ]:
# System dependencies (pyrender needs osmesa for headless rendering)
import os
import subprocess

subprocess.run(["apt-get", "install", "-qq", "libosmesa6"], check=True)
os.environ["PYOPENGL_PLATFORM"] = "osmesa"
print("System deps installed.")

In [ ]:
# Install pip packages from environment.yml (pip: section)
import subprocess
import yaml

with open("environment.yml") as f:
    env = yaml.safe_load(f)

pip_pkgs = next(
    d["pip"] for d in env["dependencies"] if isinstance(d, dict) and "pip" in d
)

print(f"Installing {len(pip_pkgs)} packages from environment.yml pip section...")
subprocess.check_call(["pip", "install", "-q"] + pip_pkgs)
print("Done.")

In [ ]:
# Install conda-listed packages not already in Colab base
# (torch/numpy/scipy/pandas/matplotlib are pre-installed on Colab)
!pip install -q \
    opencv-python-headless \
    trimesh \
    networkx \
    shapely \
    imageio \
    scikit-image \
    tqdm \
    ipywidgets
print("Conda-equivalent packages installed.")

In [ ]:
# Mount Google Drive and symlink templates/
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/autopose"
TEMPLATES_SRC = f"{DRIVE_ROOT}/templates"
TEMPLATES_DST = "/content/autopose/templates"

assert os.path.isdir(TEMPLATES_SRC), (
    f"templates/ not found at {TEMPLATES_SRC}. "
    "Upload your templates/ folder to Google Drive at MyDrive/autopose/templates/"
)

# Symlink so all downstream code uses templates/ as a normal relative path
if not os.path.islink(TEMPLATES_DST):
    os.symlink(TEMPLATES_SRC, TEMPLATES_DST)

import json
poses = json.loads(open("templates/poses.json").read())
print(f"templates/ linked. Found {len(poses)} pose entries.")

In [ ]:
# Verify GPU and key imports
import torch
import timm
import lightning
import transformers
import kornia
import roma
import trimesh
import pyrender

print(f"torch      {torch.__version__}")
print(f"lightning  {lightning.__version__}")
print(f"timm       {timm.__version__}")
print(f"transformers {transformers.__version__}")
print()

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected — go to Runtime > Change runtime type > GPU")

---

## 1 — Load Template Data

`templates/poses.json` stores 162 entries, one per icosphere vertex. Each entry has:
- `azimuth_deg` / `elevation_deg` — the viewpoint angle
- `R_world_to_cam_3x3` — rotation matrix (world → camera)
- `t_cam_to_world_3` — camera position in world space
- `file` — corresponding PNG (`tmpl_XXXX.png`)

Azimuth + elevation together define the **out-of-plane rotation** (2 of 6 DoF).

In [ ]:
# TODO: load templates/poses.json → DataFrame and visualize viewpoint sphere
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

poses = json.loads(open("templates/poses.json").read())
df = pd.DataFrame(poses)

# TODO: plot azimuth vs elevation scatter
# TODO: show a grid of template thumbnails

---

## 2 — Template Feature Extraction (Fae)

**Fae** is a DINOv2 ViT fine-tuned with InfoNCE loss (τ=0.1) so it is:
- **Sensitive** to viewpoint (azimuth + elevation)
- **Invariant** to in-plane rotation, scale, and translation

At onboarding time, all 162 templates are passed through Fae to produce cached `16×16×1024` feature maps. These are the index that Stage 1 searches over.

In [ ]:
# TODO: load pretrained DINOv2 ViT-B/14 via timm or torch.hub
# TODO: define preprocessing (resize to 224x224, normalize)
# TODO: batch forward pass over 162 template PNGs
# TODO: reshape patch tokens → (162, 16, 16, 1024)
# TODO: save to templates/template_features.pt

---

## 3 — Azimuth & Elevation Prediction (Stage 1)

Given a query image crop:
1. Encode with Fae → `(16, 16, 1024)` feature map
2. For each query patch, compute cosine similarity against all 162 × 16 × 16 cached template patches
3. Score each template by **segmentation-masked mean similarity**
4. Return the top-K template's `azimuth_deg` + `elevation_deg` from `poses.json`

No regression. The discretisation (162 icosphere bins) is the classifier.

In [ ]:
# TODO: load template_features.pt
# TODO: def retrieve_template(query_crop, fae_model, template_features, df_poses, K=1):
#           query_feat = fae_model(query_crop)           # (1, 16, 16, 1024)
#           sim = cosine_similarity(query_feat, template_features)  # (162,)
#           best_idx = sim.topk(K).indices
#           return df_poses.iloc[best_idx][['azimuth_deg', 'elevation_deg', 'R_world_to_cam_3x3']]

---

## 4 — In-Plane Refinement (Fist, Stage 2)

Once the out-of-plane rotation (az + el) is fixed from Stage 1, **Fist** (ResNet18 + MLP heads) regresses the remaining 4 DoF:
- `(cos α, sin α)` — in-plane rotation
- `(tx, ty)` — 2D translation
- `s` — scale

Input: query crop + matched template crop (concatenated along channel dim).

In [ ]:
# TODO: define Fist — ResNet18 backbone + 3 MLP heads (rotation, translation, scale)
# TODO: input: concat(query_crop, template_crop) → (B, 6, H, W)
# TODO: output: (cos_alpha, sin_alpha, tx, ty, s)

---

## 5 — Training

Two separate training runs:
1. **Fae** — InfoNCE contrastive loss. Positives: same az/el, different in-plane.
2. **Fist** — L1/L2 regression loss on `(cos α, sin α, tx, ty, s)` given Stage 1 predictions.

Use PyTorch Lightning `Trainer` with `wandb` logging.

In [ ]:
# TODO: LightningDataModule — load rendered templates + online augmentation (albumentations)
# TODO: FaeLightningModule — InfoNCE loss, configure_optimizers
# TODO: FistLightningModule — regression loss, configure_optimizers
# TODO: trainer = Trainer(max_epochs=100, accelerator='gpu', logger=WandbLogger(...))
# TODO: trainer.fit(model, datamodule)